# Hands-On 10: Build End-to-End LangGraph Agentic AI Project

This final project combines:

- Supervisor workflow
- Multiple agents
- Tools
- RAG
- Memory / checkpointer
- Human approval
- Conditional routing
- Error handling
- Optional LangSmith tracing
- Streamlit interface

## Cell 1 — Imports

In [2]:
from typing import TypedDict, Literal
import os
from datetime import datetime

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

## Cell 2 — Load `.env` and configure environment

In [3]:
load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter API key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGCHAIN_TRACING_V2", "false"))
print("LangSmith project:", os.getenv("LANGCHAIN_PROJECT", "agent_flow"))

OpenRouter API key loaded: True
LangSmith tracing: false
LangSmith project: agent_flow


## Cell 3 — Create OpenRouter LLM

In [4]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter LLM configured successfully")

OpenRouter LLM configured successfully


## Cell 4 — Test OpenRouter

In [5]:
response = llm.invoke("Explain LangGraph in one sentence.")
print(response.content)

LangGraph is a framework designed to enhance natural language processing tasks by representing language data as graphs, enabling more efficient and effective analysis and understanding of linguistic structures and relationships.


## Cell 5 — Shared State

In [6]:
class SupervisorState(TypedDict):
    query: str
    category: str
    research: str
    analysis: str
    response: str
    approved: bool
    history: list
    error: str
    retry_count: int

## Cell 6 — RAG/Search Tool

In [7]:
@tool
def search_docs(query: str) -> str:
    """Search the internal knowledge base for relevant information."""

    docs = {
        "langgraph": (
            "LangGraph is a framework for building stateful, "
            "graph-based AI agent workflows."
        ),
        "rag": (
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information and gives that context "
            "to an LLM before generating an answer."
        ),
        "agent": (
            "An AI agent combines a language model, tools, state, "
            "and workflow logic to complete tasks."
        ),
        "memory": (
            "LangGraph memory uses a checkpointer and thread ID "
            "to persist graph state across invocations."
        ),
        "human approval": (
            "Human-in-the-loop workflows pause execution and wait "
            "for a human decision before continuing."
        )
    }

    q = query.lower()
    matches = [value for key, value in docs.items() if key in q]

    return "\n".join(matches) if matches else "No relevant documents found."

## Cell 7 — Calculator Tool

In [8]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""

    allowed = set("0123456789+-*/(). ")

    if not expression or not set(expression).issubset(allowed):
        return "Error: invalid mathematical expression."

    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Calculation error: {e}"

## Cell 8 — Current Time Tool

In [9]:
@tool
def get_current_time() -> str:
    """Return the current local system time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

## Cell 9 — Test Tools

In [10]:
print("RAG:", search_docs.invoke("What is RAG?"))
print("Calculator:", calculator.invoke("25 * 4"))
print("Time:", get_current_time.invoke(""))

RAG: RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and gives that context to an LLM before generating an answer.
Calculator: 100
Time: 2026-08-21 01:37:11


## Cell 10 — Bind Tools to the Model

In [11]:
tools = [search_docs, calculator, get_current_time]
llm_with_tools = llm.bind_tools(tools)

print("Bound tools:", [t.name for t in tools])

Bound tools: ['search_docs', 'calculator', 'get_current_time']


## Cell 11 — Supervisor Classification

In [12]:
def classify_node(state: SupervisorState):
    query = state["query"].lower()

    if any(word in query for word in [
        "calculate", "math", "plus", "minus", "multiply", "divide"
    ]) or any(symbol in query for symbol in ["+", "*", "/"]):
        return {"category": "math"}

    if any(word in query for word in [
        "rag", "langgraph", "agent", "memory",
        "human approval", "what is", "explain", "research"
    ]):
        return {"category": "research"}

    return {"category": "general"}

## Cell 12 — Supervisor Router

In [13]:
def supervisor_router(state: SupervisorState) -> Literal[
    "math_node", "research_agent", "response_agent"
]:
    category = state.get("category", "general")

    if category == "math":
        return "math_node"
    if category == "research":
        return "research_agent"
    return "response_agent"

## Cell 13 — Math Agent

In [14]:
def math_node(state: SupervisorState):
    try:
        prompt = f'''
Extract only the mathematical expression from this user query.

User query:
{state["query"]}

Return ONLY the expression. Do not include explanation or markdown.
'''
        expression = llm.invoke(prompt).content.strip()
        result = calculator.invoke(expression)

        return {
            "response": f"Calculation result: {result}",
            "error": ""
        }

    except Exception as e:
        return {
            "error": f"Math agent failed: {e}"
        }

## Cell 14 — Research / RAG Agent

In [15]:
def research_agent(state: SupervisorState):
    try:
        query = state["query"]

        retrieved_context = search_docs.invoke(query)

        prompt = f'''
You are the Research/RAG Agent.

Answer the user's question using ONLY the retrieved context below.

User question:
{query}

Retrieved context:
{retrieved_context}

Rules:
- Stay grounded in the retrieved context.
- Do not invent facts.
- Give a clear and concise answer.
'''
        response = llm.invoke(prompt)

        return {
            "research": retrieved_context,
            "response": response.content,
            "error": ""
        }

    except Exception as e:
        return {
            "error": f"Research agent failed: {e}"
        }

## Cell 15 — Analysis Agent

In [16]:
def analysis_agent(state: SupervisorState):
    try:
        prompt = f'''
You are the Analysis Agent.

Review the research response below and produce a polished answer.

User query:
{state["query"]}

Retrieved research:
{state.get("research", "")}

Draft response:
{state.get("response", "")}

Improve clarity and accuracy without adding unsupported facts.
Return only the final proposed answer.
'''
        response = llm.invoke(prompt)

        return {
            "analysis": response.content,
            "response": response.content,
            "error": ""
        }

    except Exception as e:
        return {
            "error": f"Analysis agent failed: {e}"
        }

## Cell 16 — General Response Agent

In [17]:
def response_agent(state: SupervisorState):
    try:
        prompt = f'''
You are a helpful AI assistant.

User query:
{state["query"]}

Give a concise and useful response.
'''
        response = llm.invoke(prompt)

        return {
            "response": response.content,
            "error": ""
        }

    except Exception as e:
        return {
            "error": f"Response agent failed: {e}"
        }

## Cell 17 — Human Approval Node

In [18]:
def approval_node(state: SupervisorState):
    if state.get("error"):
        return {"approved": False}

    decision = interrupt({
        "question": "Approve the proposed response?",
        "query": state["query"],
        "response": state["response"]
    })

    print("Human decision:", decision)

    return {
        "approved": decision == "approved"
    }

## Cell 18 — Approval Routing and Error Handling

In [19]:
def route_after_approval(state: SupervisorState) -> Literal[
    "end", "retry_node", "error_handler"
]:
    if state.get("error"):
        return "error_handler"

    if state.get("approved"):
        return "end"

    return "retry_node"


def retry_node(state: SupervisorState):
    retry_count = state.get("retry_count", 0) + 1

    return {
        "retry_count": retry_count,
        "response": (
            "The proposed response was rejected by the human reviewer. "
            "Please review or submit the request again."
        )
    }


def error_handler_node(state: SupervisorState):
    return {
        "response": (
            "⚠️ The workflow could not complete successfully.\n\n"
            f"Reason: {state.get('error', 'Unknown error')}"
        )
    }

## Cell 19 — Build Complete Supervisor Graph

In [20]:
checkpointer = InMemorySaver()

graph = StateGraph(SupervisorState)

graph.add_node("classify", classify_node)
graph.add_node("math_node", math_node)
graph.add_node("research_agent", research_agent)
graph.add_node("analysis_agent", analysis_agent)
graph.add_node("response_agent", response_agent)
graph.add_node("approval", approval_node)
graph.add_node("retry_node", retry_node)
graph.add_node("error_handler", error_handler_node)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    supervisor_router,
    {
        "math_node": "math_node",
        "research_agent": "research_agent",
        "response_agent": "response_agent"
    }
)

graph.add_edge("math_node", "approval")
graph.add_edge("research_agent", "analysis_agent")
graph.add_edge("analysis_agent", "approval")
graph.add_edge("response_agent", "approval")

graph.add_conditional_edges(
    "approval",
    route_after_approval,
    {
        "end": END,
        "retry_node": "retry_node",
        "error_handler": "error_handler"
    }
)

graph.add_edge("retry_node", END)
graph.add_edge("error_handler", END)

app = graph.compile(checkpointer=checkpointer)

print("End-to-end LangGraph application compiled successfully")

End-to-end LangGraph application compiled successfully


## Cell 20 — Inspect Graph

In [21]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	math_node(math_node)
	research_agent(research_agent)
	analysis_agent(analysis_agent)
	response_agent(response_agent)
	approval(approval)
	retry_node(retry_node)
	error_handler(error_handler)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	analysis_agent --> approval;
	approval -. &nbsp;end&nbsp; .-> __end__;
	approval -.-> error_handler;
	approval -.-> retry_node;
	classify -.-> math_node;
	classify -.-> research_agent;
	classify -.-> response_agent;
	math_node --> approval;
	research_agent --> analysis_agent;
	response_agent --> approval;
	error_handler --> __end__;
	retry_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Cell 21 — Test Math + Human Approval

In [22]:
config1 = {
    "configurable": {
        "thread_id": "session-demo-1"
    }
}

initial_state = {
    "query": "Calculate 25 * 4",
    "category": "",
    "research": "",
    "analysis": "",
    "response": "",
    "approved": False,
    "history": [],
    "error": "",
    "retry_count": 0
}

pending = app.invoke(initial_state, config=config1)

print("Workflow paused:", "__interrupt__" in pending)

if "__interrupt__" in pending:
    print(pending["__interrupt__"][0].value)

Workflow paused: True
{'question': 'Approve the proposed response?', 'query': 'Calculate 25 * 4', 'response': 'Calculation result: 100'}


## Cell 22 — Approve Math

In [23]:
result1 = app.invoke(
    Command(resume="approved"),
    config=config1
)

print("Final response:")
print(result1["response"])

Human decision: approved
Final response:
Calculation result: 100


## Cell 23 — Test RAG + Multiple Agents

In [24]:
config2 = {
    "configurable": {
        "thread_id": "session-demo-2"
    }
}

rag_state = {
    "query": "What is RAG and how does LangGraph help build agents?",
    "category": "",
    "research": "",
    "analysis": "",
    "response": "",
    "approved": False,
    "history": [],
    "error": "",
    "retry_count": 0
}

pending2 = app.invoke(rag_state, config=config2)

print("Workflow paused:", "__interrupt__" in pending2)

if "__interrupt__" in pending2:
    print(pending2["__interrupt__"][0].value)

Workflow paused: True
{'question': 'Approve the proposed response?', 'query': 'What is RAG and how does LangGraph help build agents?', 'response': 'RAG, or Retrieval-Augmented Generation, is a technique that retrieves relevant information and supplies that context to a language model (LLM) before generating a response. LangGraph is a framework designed to facilitate the creation of stateful, graph-based AI agent workflows. It integrates a language model, various tools, state management, and workflow logic to effectively complete tasks.'}


## Cell 24 — Approve RAG

In [25]:
result2 = app.invoke(
    Command(resume="approved"),
    config=config2
)

print("Retrieved research:")
print(result2["research"])

print("\nFinal response:")
print(result2["response"])

Human decision: approved
Retrieved research:
LangGraph is a framework for building stateful, graph-based AI agent workflows.
RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and gives that context to an LLM before generating an answer.
An AI agent combines a language model, tools, state, and workflow logic to complete tasks.

Final response:
RAG, or Retrieval-Augmented Generation, is a technique that retrieves relevant information and supplies that context to a language model (LLM) before generating a response. LangGraph is a framework designed to facilitate the creation of stateful, graph-based AI agent workflows. It integrates a language model, various tools, state management, and workflow logic to effectively complete tasks.


## Cell 25 — Test Rejection

In [26]:
config3 = {
    "configurable": {
        "thread_id": "session-demo-3"
    }
}

reject_state = {
    "query": "Explain LangGraph",
    "category": "",
    "research": "",
    "analysis": "",
    "response": "",
    "approved": False,
    "history": [],
    "error": "",
    "retry_count": 0
}

pending3 = app.invoke(reject_state, config=config3)

print(pending3["__interrupt__"][0].value)

{'question': 'Approve the proposed response?', 'query': 'Explain LangGraph', 'response': 'LangGraph is a framework specifically designed for developing stateful workflows for AI agents that utilize graph-based structures.'}


## Cell 26 — Reject

In [27]:
result3 = app.invoke(
    Command(resume="rejected"),
    config=config3
)

print(result3["response"])
print("Approved:", result3["approved"])

Human decision: rejected
The proposed response was rejected by the human reviewer. Please review or submit the request again.
Approved: False


## Cell 27 — Test Memory / Same Thread

In [28]:
memory_state = app.get_state(config1)

print("Thread ID:", config1["configurable"]["thread_id"])
print("Stored state:")
for key, value in memory_state.values.items():
    print(f"{key}: {value}")

Thread ID: session-demo-1
Stored state:
query: Calculate 25 * 4
category: math
research: 
analysis: 
response: Calculation result: 100
approved: True
history: []
error: 
retry_count: 0


## Cell 28 — Test Error Handler

In [29]:
error_test = error_handler_node({
    "error": "Simulated workflow failure"
})

print(error_test["response"])

⚠️ The workflow could not complete successfully.

Reason: Simulated workflow failure


## Cell 29 — LangSmith Trace Configuration

In [30]:
print("LANGCHAIN_TRACING_V2 =", os.getenv("LANGCHAIN_TRACING_V2", "false"))
print("LANGCHAIN_PROJECT =", os.getenv("LANGCHAIN_PROJECT", "agent_flow"))

print(
    "\nTracing is optional. To enable it, put a valid "
    "LANGCHAIN_API_KEY in your .env and set LANGCHAIN_TRACING_V2=true."
)

LANGCHAIN_TRACING_V2 = false
LANGCHAIN_PROJECT = agent_flow

Tracing is optional. To enable it, put a valid LANGCHAIN_API_KEY in your .env and set LANGCHAIN_TRACING_V2=true.


## Cell 30 — Final Verification

In [31]:
print("=== HANDS-ON 10 COMPLETE ===")
print("✓ OpenRouter LLM")
print("✓ Supervisor workflow")
print("✓ Multiple agents")
print("✓ Tools")
print("✓ RAG")
print("✓ Memory / checkpointer")
print("✓ Human approval")
print("✓ Conditional routing")
print("✓ Error handling")
print("✓ Optional LangSmith tracing")
print("✓ Streamlit application prepared")
print("✓ Math, RAG, approval/rejection, memory tests")

=== HANDS-ON 10 COMPLETE ===
✓ OpenRouter LLM
✓ Supervisor workflow
✓ Multiple agents
✓ Tools
✓ RAG
✓ Memory / checkpointer
✓ Human approval
✓ Conditional routing
✓ Error handling
✓ Optional LangSmith tracing
✓ Streamlit application prepared
✓ Math, RAG, approval/rejection, memory tests
